# NB12 – Pairing Concept Figure (v4 — final)

Fetches **actual images** from HF and renders a clean paper figure.

**Layout:**
```
         [ Real photograph  512×512 ]
    BLIP-2 caption:  "a group of people..."
           ↓  ↓  ↓  ↓  ↓  ↓
  [SD 1.5][SDXL][FLUX][Kand.][PixArt][Cascade]
```

### Only edit Cell 2 (CONFIG)
| Variable | Default | Effect |
|---|---|---|
| `SOURCE_REAL_ID` | `None` | Pin a pair e.g. `'real_000042'`; `None` = random |
| `SPLIT` | `'test'` | `train` / `val` / `test` |
| `RANDOM_SEED` | `42` | Change integer = different pair |
| `REAL_SIZE` | `320` | Real image tile size (px) |
| `AI_SIZE` | `210` | Each AI tile size (px) |
| `FIGURE_DPI` | `220` | Output DPI |
| `OUTPUT_STEM` | `'/kaggle/working/pairing_concept'` | Path, no extension |
| `PUSH_TO_HF` | `False` | Push PNG+PDF to HF |

**Prerequisites:** NB00–NB11 complete. No GPU. Internet ON.

**Quick re-render:** change `QUICK_SEED` in last cell → run it → re-run Cell 7 → Cell 8.

In [ ]:
import importlib, subprocess, sys

def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

need = []
for mod, pkg in [
    ('huggingface_hub', 'huggingface_hub>=0.23'),
    ('pyarrow',         'pyarrow'),
    ('PIL',             'pillow'),
    ('matplotlib',      'matplotlib>=3.7'),
]:
    if importlib.util.find_spec(mod) is None:
        need.append(pkg)
if need:
    _pip(*need)
    print('installed:', need)
else:
    print('all deps present')

import matplotlib
print(f'matplotlib {matplotlib.__version__}')

In [ ]:
# ── CONFIG — only cell you need to edit ──────────────────────────────────────

REPO_ID = 'Shanmuk4622/ai-detection-dataset-v2'

SOURCE_REAL_ID = None      # None = random; or e.g. 'real_000042'
SPLIT          = 'test'
RANDOM_SEED    = 42

REAL_SIZE   = 320   # real image display size (px)
AI_SIZE     = 210   # each AI image display size (px)
FIGURE_DPI  = 220
OUTPUT_STEM = '/kaggle/working/pairing_concept'
PUSH_TO_HF  = False

GEN_COLORS = {
    'sd15':         '#3B82F6',
    'sdxl':         '#8B5CF6',
    'flux_schnell': '#10B981',
    'kandinsky22':  '#F59E0B',
    'pixart_sigma': '#EF4444',
    'wuerstchen':   '#6B7280',
}
GEN_LABELS = {
    'sd15':         'SD 1.5',
    'sdxl':         'SDXL',
    'flux_schnell': 'FLUX.1-schnell',
    'kandinsky22':  'Kandinsky 2.2',
    'pixart_sigma': 'PixArt-Σ',
    'wuerstchen':   'Würstchen',
}

print('Config OK.')

In [ ]:
import os, io, random
from huggingface_hub import HfApi, hf_hub_download, list_repo_files
import pyarrow.parquet as pq
from PIL import Image

def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret('HF_TOKEN')
        if t: return t.strip()
    except Exception: pass
    for k in ('HF_TOKEN', 'HUGGINGFACE_TOKEN'):
        v = os.environ.get(k)
        if v: return v.strip()
    import getpass; return getpass.getpass('HF token: ').strip()

TOKEN = load_token()
os.environ['HF_TOKEN'] = TOKEN
api = HfApi(token=TOKEN)

lib  = hf_hub_download(REPO_ID, 'ai_detect_common.py', repo_type='dataset', token=TOKEN)
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location('ai_detect_common', lib)
C     = _ilu.module_from_spec(_spec); _spec.loader.exec_module(C)
cfg  = C.read_config(REPO_ID, TOKEN)
GENERATORS = list(cfg['generators'].keys())
gen_order  = {g: i for i, g in enumerate(['real'] + GENERATORS)}
print(f'Library {C.PIPELINE_VERSION} | generators: {GENERATORS}')

In [ ]:
# manifest columns: image_id, source_real_id, generator, label, sha256, split
# 'prompt' is NOT in the manifest — it lives in captions/*.parquet

man_path = hf_hub_download(REPO_ID, 'manifest.parquet',
                           repo_type='dataset', token=TOKEN)
manifest = pq.read_table(man_path).to_pydict()
print(f'Manifest: {len(manifest["image_id"])} rows | cols: {list(manifest.keys())}')

if SOURCE_REAL_ID is not None:
    if SOURCE_REAL_ID not in set(manifest['source_real_id']):
        raise ValueError(f'{SOURCE_REAL_ID!r} not found in manifest')
    chosen_id = SOURCE_REAL_ID
    print(f'Pinned pair: {chosen_id!r}')
else:
    _rng  = random.Random(RANDOM_SEED)
    _pool = sorted(set(
        sid for sid, sp in zip(manifest['source_real_id'], manifest['split'])
        if sp == SPLIT
    ))
    chosen_id = _rng.choice(_pool)
    print(f'Random pair (seed={RANDOM_SEED}, split={SPLIT!r}): {chosen_id!r}')

# pair_rows: (image_id, generator, label) — NO prompt
pair_rows = sorted(
    [(iid, gen, lbl)
     for iid, gen, lbl, sid in zip(
         manifest['image_id'], manifest['generator'],
         manifest['label'],    manifest['source_real_id'])
     if sid == chosen_id],
    key=lambda x: gen_order.get(x[1], 99)
)
print(f'Pair has {len(pair_rows)} rows:')
for r in pair_rows:
    print(f'  {r[0]:25s}  gen={r[1]:15s}  label={r[2]}')

In [ ]:
# captions/*.parquet — tiny (~24KB each), text only
# columns: source_real_id, caption, n_tokens

print('Building repo file list...')
all_repo_files = sorted(list_repo_files(REPO_ID, repo_type='dataset', token=TOKEN))
print(f'  {len(all_repo_files)} files in repo')

cap_files = [f for f in all_repo_files
             if f.startswith('captions/') and f.endswith('.parquet')]
print(f'  {len(cap_files)} caption shards')

prompt_text = '(caption not found)'
for cf in cap_files:
    local = hf_hub_download(REPO_ID, cf, repo_type='dataset', token=TOKEN)
    t    = pq.read_table(local, columns=['source_real_id', 'caption'])
    ids  = t.column('source_real_id').to_pylist()
    if chosen_id in ids:
        prompt_text = t.column('caption')[ids.index(chosen_id)].as_py() or '(empty)'
        print(f'Caption found in {cf.split("/")[-1]}')
        break
else:
    print(f'WARNING: caption not found for {chosen_id!r}')

print(f'Caption: {prompt_text!r}')

In [ ]:
# Build shard lists per generator prefix — from all_repo_files (already fetched,
# no extra downloads). Real images are under 'real/', AI under 'data/{gen}/'.

def _shards_for(generator):
    """Return sorted list of parquet shard paths for a generator."""
    prefix = 'real/' if generator == 'real' else f'data/{generator}/'
    return sorted(f for f in all_repo_files
                  if f.startswith(prefix) and f.endswith('.parquet'))

# Print shard counts so we can verify the prefix is correct
print('Shard counts per prefix:')
print(f"  real/  -> {len(_shards_for('real'))} shards")
for g in GENERATORS:
    print(f'  data/{g}/  -> {len(_shards_for(g))} shards')

# Two-level cache
_id_cache  = {}   # shard_path -> list[str]   (image_id column only)
_img_cache = {}   # shard_path -> pa.Table    (image_id + image bytes)

def _find_shard(image_id, generator):
    """Scan shards for the given generator prefix until image_id is found.
    Downloads only the image_id column — not image bytes — for each new shard."""
    for sf in _shards_for(generator):
        if sf not in _id_cache:
            loc = hf_hub_download(REPO_ID, sf, repo_type='dataset', token=TOKEN)
            _id_cache[sf] = (pq.read_table(loc, columns=['image_id'])
                               .column('image_id').to_pylist())
        if image_id in _id_cache[sf]:
            return sf
    raise FileNotFoundError(
        f"'{image_id}' not found in any shard for generator='{generator}'.\n"
        f"Prefix searched: {'real/' if generator=='real' else f'data/{generator}/'}\n"
        f"Shards checked : {_shards_for(generator)[:3]} ..."
    )

def _get_image(image_id, generator):
    """Return PIL Image. Downloads full shard only once, cached."""
    sf = _find_shard(image_id, generator)
    if sf not in _img_cache:
        loc = hf_hub_download(REPO_ID, sf, repo_type='dataset', token=TOKEN)
        _img_cache[sf] = pq.read_table(loc, columns=['image_id', 'image'])
        print(f'  [img shard] {sf}')
    tbl = _img_cache[sf]
    row = tbl.column('image_id').to_pylist().index(image_id)
    return Image.open(io.BytesIO(tbl.column('image')[row].as_py())).convert('RGB')

print('\nFetching 7 images...')
pair_images = []   # (image_id, generator, label, PIL.Image)
for iid, gen, lbl in pair_rows:
    img = _get_image(iid, gen)
    pair_images.append((iid, gen, lbl, img))
    print(f'  OK  {iid:25s}  gen={gen:15s}  label={lbl}  size={img.size}')

real_entry = next(e for e in pair_images if e[2] == 0)
ai_entries = [e for e in pair_images if e[2] == 1]
print(f'\nReady: 1 real + {len(ai_entries)} AI images.')
print(f'Caption: {prompt_text!r}')

In [ ]:
# ── RENDER ───────────────────────────────────────────────────────────────────
#
# Clean paper figure. Layout (top to bottom):
#
#   SOURCE label   (small italic)
#   [ Real photograph — large, centred ]
#   BLIP-2 caption box
#   Five downward arrows
#   [ SD1.5 ] [ SDXL ] [ FLUX ] [ Kand ] [ PixArt ] [ Würstchen ]
#   Generator name labels
#
# Design rules:
#  - Real image larger than AI tiles (it is the anchor)
#  - NO preprocessing box inside the figure (it belongs in the LaTeX caption)
#  - NO 'label=0/1' clutter — REAL/AI badges in image corners only
#  - Generator name is the only label below each AI tile
#  - White background, minimal chrome
#  - All positions set with fig.add_axes for pixel-precise control

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import textwrap

plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'font.size':          9,
    'savefig.dpi':        FIGURE_DPI,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.08,
    'figure.facecolor':   'white',
})

def _arr(pil_img, size):
    """Resize PIL image to square and return uint8 numpy array."""
    return np.asarray(pil_img.resize((size, size), Image.LANCZOS))

N = len(ai_entries)   # 6

# ── Figure geometry (all in inches) ─────────────────────────────────────────
FIG_W    = 10.0          # total figure width
MARGIN   = 0.25          # left/right margin

real_w   = REAL_SIZE / FIGURE_DPI * (FIG_W / 4.5)   # scale real tile to fig
real_h   = real_w                                     # square
ai_w     = (FIG_W - 2*MARGIN) / N - 0.12            # AI tiles fill width
ai_h     = ai_w                                      # square

caption_h = 0.65    # space for caption text box
arrow_h   = 0.35    # space for arrows
label_h   = 0.32    # space for generator name labels below AI tiles
source_h  = 0.22    # space for source label above real image

FIG_H = source_h + real_h + caption_h + arrow_h + ai_h + label_h

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')

# Convert inches to figure fractions
def _fx(x_in):  return x_in / FIG_W
def _fy(y_in):  return y_in / FIG_H

# Y positions from bottom
y_label   = 0.0
y_ai      = y_label   + label_h
y_arrow   = y_ai      + ai_h
y_caption = y_arrow   + arrow_h
y_real    = y_caption + caption_h
y_source  = y_real    + real_h

# ── Real image ───────────────────────────────────────────────────────────────
real_x_in = (FIG_W - real_w) / 2   # centred
ax_real = fig.add_axes([_fx(real_x_in), _fy(y_real), _fx(real_w), _fy(real_h)])
ax_real.imshow(_arr(real_entry[3], REAL_SIZE), interpolation='lanczos')
ax_real.set_xticks([]); ax_real.set_yticks([])
for sp in ax_real.spines.values():
    sp.set_visible(True); sp.set_edgecolor('#1e3a5f'); sp.set_linewidth(2.5)

# REAL badge (top-left corner of image)
ax_real.text(0.03, 0.97, 'REAL', transform=ax_real.transAxes,
             ha='left', va='top', fontsize=8, fontweight='bold', color='white',
             bbox=dict(boxstyle='round,pad=0.28', fc='#1e3a5f', ec='none', alpha=0.88))

# Source label above image
fig.text(_fx(FIG_W/2), _fy(y_source + source_h*0.35),
         f'Source photograph  —  {real_entry[0]}',
         ha='center', va='center', fontsize=8.2, color='#6b7280', style='italic')

# ── Caption box ──────────────────────────────────────────────────────────────
wrapped = textwrap.fill(f'“{prompt_text}”', width=72)

# 'BLIP-2 caption' micro-label above the box
fig.text(_fx(FIG_W/2), _fy(y_caption + caption_h*0.95),
         'BLIP-2 caption  →  passed unchanged to all 6 generators',
         ha='center', va='top', fontsize=8.0, color='#6b7280')

# Caption text box
fig.text(_fx(FIG_W/2), _fy(y_caption + caption_h*0.56),
         wrapped,
         ha='center', va='center', fontsize=9.2, color='#1d4ed8', style='italic',
         bbox=dict(boxstyle='round,pad=0.45', fc='#eff6ff', ec='#93c5fd', lw=1.1))

# ── Arrows ────────────────────────────────────────────────────────────────────
ax_arr = fig.add_axes([_fx(MARGIN*2), _fy(y_arrow), _fx(FIG_W - 4*MARGIN), _fy(arrow_h)])
ax_arr.set_xlim(0, 1); ax_arr.set_ylim(0, 1); ax_arr.axis('off')
for xfrac in np.linspace(0.08, 0.92, N):
    ax_arr.annotate(
        '', xy=(xfrac, 0.05), xytext=(xfrac, 0.92),
        arrowprops=dict(arrowstyle='->', color='#9ca3af', lw=1.4, mutation_scale=11)
    )

# ── Six AI images ─────────────────────────────────────────────────────────────
gap      = 0.10                              # gap between tiles (inches)
total_w  = FIG_W - 2*MARGIN
tile_w   = (total_w - (N-1)*gap) / N
tile_h   = tile_w

for idx, (iid, gen, lbl, img) in enumerate(ai_entries):
    x_in  = MARGIN + idx * (tile_w + gap)
    color = GEN_COLORS.get(gen, '#888888')
    name  = GEN_LABELS.get(gen, gen)

    ax = fig.add_axes([_fx(x_in), _fy(y_ai), _fx(tile_w), _fy(tile_h)])
    ax.imshow(_arr(img, AI_SIZE), interpolation='lanczos')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_edgecolor(color); sp.set_linewidth(2.0)

    # AI badge top-left
    ax.text(0.04, 0.96, 'AI', transform=ax.transAxes,
            ha='left', va='top', fontsize=7.5, fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.22', fc=color, ec='none', alpha=0.88))

    # Generator name below tile
    fig.text(_fx(x_in + tile_w/2), _fy(y_label + label_h*0.55),
             name, ha='center', va='center',
             fontsize=8.8, fontweight='500', color=color)

print('Figure rendered. Run next cell to save.')

In [ ]:
from pathlib import Path

for ext in ['png', 'pdf', 'svg']:
    out = f'{OUTPUT_STEM}.{ext}'
    plt.savefig(out, format=ext, facecolor='white')
    print(f'Saved  {out}  ({Path(out).stat().st_size // 1024} KB)')

plt.show()
print('Done.')

In [ ]:
if PUSH_TO_HF:
    from huggingface_hub import CommitOperationAdd
    from pathlib import Path
    ops = []
    for ext in ['png', 'pdf']:
        lp = f'{OUTPUT_STEM}.{ext}'
        rp = f'results/figures/pairing_concept_{chosen_id}.{ext}'
        if Path(lp).exists():
            ops.append(CommitOperationAdd(path_in_repo=rp, path_or_fileobj=lp))
    if ops:
        api.create_commit(
            repo_id=REPO_ID, repo_type='dataset', operations=ops,
            commit_message=f'NB12 v4: pairing figure ({chosen_id})'
        )
        print('Pushed:', [op.path_in_repo for op in ops])
    else:
        print('No output files found.')
else:
    print('PUSH_TO_HF=False  ->  set True in Cell 2 to upload.')

## Quick re-render

Change `QUICK_SEED` below → run this cell → re-run **Cell 7 (render)** → **Cell 8 (save)**.

Shard caches `_id_cache` and `_img_cache` are reused — only shards not yet downloaded will be fetched.

In [ ]:
# ── Quick re-render shortcut ──────────────────────────────────────────────────
# Change QUICK_SEED → run this cell → re-run Cell 6 (render) → Cell 7 (save)
QUICK_SEED = 100   # <-- change this

import random, io
import pyarrow.parquet as pq
from PIL import Image
from huggingface_hub import hf_hub_download, list_repo_files

_rng      = random.Random(QUICK_SEED)
_pool     = sorted(set(
    sid for sid, sp in zip(manifest['source_real_id'], manifest['split'])
    if sp == SPLIT
))
chosen_id = _rng.choice(_pool)
print(f'Selected: {chosen_id!r}  (QUICK_SEED={QUICK_SEED})')

# ── pair_rows: image_id, generator, label — NO prompt from manifest ───────────
pair_rows = sorted([
    (iid, gen, lbl)
    for iid, gen, lbl, sid in zip(
        manifest['image_id'], manifest['generator'],
        manifest['label'],    manifest['source_real_id']
    )
    if sid == chosen_id
], key=lambda x: gen_order.get(x[1], 99))

# ── fetch caption from captions/ (not from manifest) ─────────────────────────
_all_files = sorted(list_repo_files(REPO_ID, repo_type='dataset', token=TOKEN))
_cap_files = [f for f in _all_files if f.startswith('captions/') and f.endswith('.parquet')]

prompt_text = '(caption not found)'
for _cf in _cap_files:
    _loc = hf_hub_download(REPO_ID, _cf, repo_type='dataset', token=TOKEN)
    _t   = pq.read_table(_loc, columns=['source_real_id', 'caption'])
    _ids = _t.column('source_real_id').to_pylist()
    if chosen_id in _ids:
        prompt_text = _t.column('caption')[_ids.index(chosen_id)].as_py() or '(empty)'
        break
print(f'Caption: {prompt_text!r}')

# ── shard helpers (defined here so they work without v4 cell 6) ──────────────
_all_files_set = _all_files   # already fetched above

def _shards_for(generator):
    prefix = 'real/' if generator == 'real' else f'data/{generator}/'
    return sorted(f for f in _all_files_set
                  if f.startswith(prefix) and f.endswith('.parquet'))

_id_cache  = {}
_img_cache = {}

def _get_img(image_id, generator):
    for sf in _shards_for(generator):
        if sf not in _id_cache:
            loc = hf_hub_download(REPO_ID, sf, repo_type='dataset', token=TOKEN)
            _id_cache[sf] = pq.read_table(loc, columns=['image_id']).column('image_id').to_pylist()
        if image_id in _id_cache[sf]:
            if sf not in _img_cache:
                loc = hf_hub_download(REPO_ID, sf, repo_type='dataset', token=TOKEN)
                _img_cache[sf] = pq.read_table(loc, columns=['image_id', 'image'])
                print(f'  shard: {sf.split("/")[-1]}')
            tbl = _img_cache[sf]
            row = tbl.column('image_id').to_pylist().index(image_id)
            return Image.open(io.BytesIO(tbl.column('image')[row].as_py())).convert('RGB')
    raise FileNotFoundError(f'{image_id!r} not found for generator={generator!r}')

# ── fetch all 7 images ────────────────────────────────────────────────────────
print('Fetching images...')
pair_images = []
for iid, gen, lbl in pair_rows:
    img = _get_img(iid, gen)
    pair_images.append((iid, gen, lbl, prompt_text if lbl == 1 else None, img))
    print(f'  {iid:25s}  {gen:15s}  {img.size}')

real_e   = next((e for e in pair_images if e[2] == 0), None)
ai_e     = [e for e in pair_images if e[2] == 1]
import numpy as np
from PIL import Image as _Image

_SZ = 224  # same as DISPLAY_SIZE in CONFIG — change if you changed it there

def pil_to_arr(pil_img):
    return np.asarray(pil_img.resize((_SZ, _SZ), _Image.LANCZOS)) / 255.0

real_arr = pil_to_arr(real_e[4])
ai_arrs  = [(e[1], pil_to_arr(e[4])) for e in ai_e]
print(f'\nDone. Re-run the render+save cell.')
print(f'\nDone. Re-run Cell 6 (render) then Cell 7 (save).')
print(f'\nDone. Re-run Cell 6 (render) then Cell 7 (save).')

In [ ]:
# ── Re-render + save ─────────────────────────────────────────────────────────
# Self-contained. Only needs: real_e, ai_e, prompt_text, chosen_id,
#                             SPLIT, QUICK_SEED  (all set by the fetch cell)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, textwrap
from pathlib import Path
from PIL import Image as _PIL

# ── Constants ─────────────────────────────────────────────────────────────────
_SZ         = 224
_FIG_W      = 14.5
_FIG_DPI    = 220
_OUT        = '/kaggle/working/pairing_concept'

_COLORS = {
    'sd15':         '#3B82F6',
    'sdxl':         '#8B5CF6',
    'flux_schnell': '#10B981',
    'kandinsky22':  '#F59E0B',
    'pixart_sigma': '#EF4444',
    'wuerstchen':   '#6B7280',
}
_NAMES = {
    'sd15':         'SD 1.5',
    'sdxl':         'SDXL',
    'flux_schnell': 'FLUX.1-schnell',
    'kandinsky22':  'Kandinsky 2.2',
    'pixart_sigma': 'PixArt-\u03a3',
    'wuerstchen':   'Würstchen',
}

plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'savefig.dpi':        _FIG_DPI,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.06,
    'figure.facecolor':   'white',
})

def _arr(pil_img):
    return np.asarray(pil_img.resize((_SZ, _SZ), _PIL.LANCZOS))

# ── Figure geometry (all in inches) ──────────────────────────────────────────
_CAP_H = 0.50    # caption bar height
_IMG_H = 2.90    # image row height
_LAB_H = 0.42    # label row height
_FIG_H = _CAP_H + _IMG_H + _LAB_H + 0.10

fig = plt.figure(figsize=(_FIG_W, _FIG_H), facecolor='white')

def _fy(y): return y / _FIG_H
def _fh(h): return h / _FIG_H
def _fx(x): return x / _FIG_W
def _fw(w): return w / _FIG_W

_y_lab = 0.03
_y_img = _y_lab + _LAB_H
_y_cap = _y_img + _IMG_H + 0.04

_LMARGIN = 0.10
_RMARGIN = 0.10
_SEP     = 0.18   # gap between real image and first AI
_IGAP    = 0.06   # gap between AI images

_total_x  = _FIG_W - _LMARGIN - _RMARGIN - _SEP
_real_w   = _total_x * 0.165
_ai_total = _total_x - _real_w
_ai_w     = (_ai_total - 5 * _IGAP) / 6

# ── Caption bar ───────────────────────────────────────────────────────────────
ax_cap = fig.add_axes([_fx(_LMARGIN), _fy(_y_cap),
                        _fw(_FIG_W - _LMARGIN - _RMARGIN), _fh(_CAP_H)])
ax_cap.set_xlim(0, 1); ax_cap.set_ylim(0, 1); ax_cap.axis('off')
ax_cap.add_patch(mpatches.FancyBboxPatch(
    (0, 0.05), 1.0, 0.92, boxstyle='round,pad=0.01',
    fc='#f8fafc', ec='#e2e8f0', lw=0.8,
    transform=ax_cap.transAxes, clip_on=False
))
ax_cap.text(0.008, 0.50,
            'BLIP-2 caption \u2014 passed unchanged to all 6 generators:',
            ha='left', va='center', fontsize=8.5, color='#64748b', style='italic')
ax_cap.text(0.32, 0.50,
            textwrap.fill(f'\u201c{prompt_text}\u201d', width=100),
            ha='left', va='center', fontsize=9.0,
            color='#1d4ed8', style='italic', fontweight='500')

# ── Real image ────────────────────────────────────────────────────────────────
_x_real = _LMARGIN
ax_r = fig.add_axes([_fx(_x_real), _fy(_y_img), _fw(_real_w), _fh(_IMG_H)])
ax_r.imshow(_arr(real_e[4]), interpolation='lanczos')
ax_r.set_xticks([]); ax_r.set_yticks([])
for sp in ax_r.spines.values():
    sp.set_visible(True); sp.set_edgecolor('#1e3a5f'); sp.set_linewidth(2.8)
ax_r.text(0.04, 0.965, 'Real', transform=ax_r.transAxes,
          ha='left', va='top', fontsize=9, fontweight='bold', color='white',
          bbox=dict(boxstyle='square,pad=0.30', fc='#1e3a5f', ec='none', alpha=0.92))
fig.text(_fx(_x_real + _real_w / 2), _fy(_y_lab + _LAB_H * 0.48),
         'Source photograph',
         ha='center', va='center', fontsize=9.0, fontweight='700', color='#1e3a5f')

# Thin vertical separator between real and AI
fig.add_artist(plt.matplotlib.lines.Line2D(
    [_fx(_x_real + _real_w + _SEP / 2)] * 2,
    [_fy(_y_img + 0.04), _fy(_y_img + _IMG_H - 0.04)],
    transform=fig.transFigure, color='#cbd5e1', lw=1.0, clip_on=False
))

# ── Six AI images ─────────────────────────────────────────────────────────────
_x0 = _x_real + _real_w + _SEP
for _i, (_iid, _gen, _lbl, _prm, _img) in enumerate(ai_e):
    _x     = _x0 + _i * (_ai_w + _IGAP)
    _color = _COLORS.get(_gen, '#888888')
    ax = fig.add_axes([_fx(_x), _fy(_y_img), _fw(_ai_w), _fh(_IMG_H)])
    ax.imshow(_arr(_img), interpolation='lanczos')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_edgecolor(_color); sp.set_linewidth(2.0)
    ax.text(0.04, 0.965, 'AI', transform=ax.transAxes,
            ha='left', va='top', fontsize=8, fontweight='bold', color='white',
            bbox=dict(boxstyle='square,pad=0.25', fc=_color, ec='none', alpha=0.92))
    fig.text(_fx(_x + _ai_w / 2), _fy(_y_lab + _LAB_H * 0.48),
             _NAMES.get(_gen, _gen),
             ha='center', va='center', fontsize=8.8,
             fontweight='600', color=_color)

# ── Footer ────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.002,
         f'source_real_id: {chosen_id}  \u2502  split: {SPLIT}  \u2502  QUICK_SEED: {QUICK_SEED}',
         ha='center', va='bottom', fontsize=7.0, color='#94a3b8')

# ── Save ──────────────────────────────────────────────────────────────────────
for _ext in ['png', 'pdf', 'svg']:
    _out = f'{_OUT}.{_ext}'
    plt.savefig(_out, format=_ext, facecolor='white')
    print(f'Saved  {_out}  ({Path(_out).stat().st_size // 1024} KB)')

plt.show()
print('Done.')